# 第 2 周练习：API 文档问答

## 练习目标

做一个小型 **API 文档问答助手**：

- 用轻量 **检索工具**（关键词重叠）从文档块里取出相关段落
- 模型通过 **Function Calling** 调用 `search_docs`
- 强制用固定标题结构回答（简答 / 细节 / 引文）

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tool Use | `TOOLS` + `tool_choice='auto'` |
| 检索当工具 | `search_docs` 不是向量库，是简单词频打分 |
| 结构化输出 | system prompt 规定固定 Markdown 标题 |
| OpenRouter | `base_url='https://openrouter.ai/api/v1'` |

## 怎么跑

1. `.env` 写入 `OPENAI_API_KEY`（本笔记本经 OpenRouter 使用）
2. 依次运行导入 → 密钥检查 → 文档/分块/工具 → `ask(...)`
3. 最后一格有两个英文示例问题；可改问题再跑


In [1]:
# ========== 导入：环境、正则、JSON、OpenAI ==========

# 标准库 os：读环境变量里的 API Key
import os
# 标准库 re：检索时用正则拆词（findall \w+）
import re
# 标准库 json：解析 tool_call.function.arguments
import json
# 从 dotenv 导入 load_dotenv：加载 .env
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端（后面指向 OpenRouter）
from openai import OpenAI


In [2]:
# ========== 环境变量 + OpenRouter 客户端 ==========

# override=True：.env 覆盖进程里已有同名环境变量（保持原参数）
load_dotenv(override=True)
# 读取 OPENAI_API_KEY（本练习仍用这个变量名，即使走 OpenRouter）
api_key = os.getenv('OPENAI_API_KEY')

# 三种密钥自检：缺失 / 首尾空白 / 看起来正常（print 文案保持英文）
if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
else:
    print('API key looks good!')

# 变量名 openai：OpenAI 客户端实例；base_url 指向 OpenRouter 兼容端点
openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)


API key looks good!


In [3]:
# ========== 模型选型：集中写常量，后面只改这里 ==========

# 模型 id 字符串；经 OpenRouter 路由，勿擅自改名除非你知道账号可用模型
MODEL = 'gpt-4.1-mini'


In [12]:
# ========== 示例 API 文档语料（可换成你自己的文档）==========

# 三引号大字符串：既是「知识库」原文，也会被 chunk / search 使用
# 正文里中英混排与路径字面量保持原样——改写会影响检索命中与回答依据
API_DOCS = '''
# Acme 支付 API

# 验证
Use a bearer token in the Authorization header. Tokens expire after 24 hours.

# 创建费用（POST /v1/费用）
Required fields: amount (integer, cents), currency (string), source (string token).
Optional: description, metadata.
Returns: charge_id, status, created_at.

# 退款费用（POST /v1/charges/{charge_id}/refunds）
Required fields: amount (integer, cents).
Optional: reason (string).
Returns: refund_id, status.

# 列出费用（GET /v1/charges）
Query params: status, limit, starting_after.
Returns a paginated list of charges.

# 网络钩子
We send events for charge.succeeded, charge.failed, refund.created.
Retry policy: 3 attempts over 24 hours.
'''


In [13]:
# ========== 分块：按 Markdown 标题 # 切开文档 ==========

# 简单规则：遇到新的以 # 开头的行，就结束上一段并开新段
def chunk_docs(text: str):
    # chunks：已完成的段落列表；current：正在积累的行
    chunks = []
    current = []
    # 按行扫描全文
    for line in text.splitlines():
        if line.startswith('#'):  # new section
            # 若当前段非空，先收束进 chunks
            if current:
                chunks.append('\n'.join(current).strip())
                current = []
        # 标题行本身也属于新段的第一行
        current.append(line)
    # 文件末尾最后一段
    if current:
        chunks.append('\n'.join(current).strip())
    # 丢掉空串，得到干净块列表
    return [c for c in chunks if c]

# 对示例文档跑一遍，得到全局 DOC_CHUNKS 供检索使用
DOC_CHUNKS = chunk_docs(API_DOCS)


In [14]:
# ========== 检索工具：关键词重叠打分，取 top-k ==========

# query：用户/模型传来的查询；k：最多返回几段（默认 3）
def search_docs(query: str, k: int = 3):
    # 小写后用 \w+ 抽词；没有词就无法打分
    q = re.findall(r'\w+', query.lower())
    if not q:
        return []
    # (score, chunk) 列表，后面按分数排序
    scores = []
    for chunk in DOC_CHUNKS:
        # 段落也转小写再 count
        text = chunk.lower()
        # 简单打分：查询词在段落中出现次数之和
        score = sum(text.count(word) for word in q)
        scores.append((score, chunk))
    # 分数从高到低
    scores.sort(key=lambda x: x[0], reverse=True)
    # 只保留 score>0 的前 k 段
    return [c for s, c in scores if s > 0][:k]


In [15]:
# ========== TOOLS：把 search_docs 暴露给模型 ==========

# OpenAI tools schema；description/参数名保持英文，影响模型是否调用
TOOLS = [
    {
        'type': 'function',
        'function': {
            # 与本地函数同名，便于分发
            'name': 'search_docs',
            'description': 'Search the API docs and return relevant sections.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string'},
                    # k 可选，默认 3（schema 里的 default 给模型参考）
                    'k': {'type': 'integer', 'default': 3},
                },
                'required': ['query']
            }
        }
    }
]


In [16]:
# ========== SYSTEM_PROMPT：强制固定回答标题结构 ==========

# 整段英文+标题要求发给模型；示例里的「# 简答」等标题字面量勿改
SYSTEM_PROMPT = '''
You are an API documentation assistant.
When needed, call the search_docs tool to retrieve relevant sections.
You MUST respond using these exact headings (with ##):
# 简答
# 细节
# 引文
Under Citations, use bullet points and quote the section titles used.
If the docs don't mention it, say so clearly.
If you fail to use the exact headings, the answer is invalid.
Example format:
# 简答
...
# 细节
...
# 引文
- "Section Title"
'''


In [17]:
# ========== ask：一轮（或带一次工具回传）问答 ==========

# question：用户自然语言问题；返回模型最终 content 字符串
def ask(question: str):
    # 初始 messages：system 定格式，user 放问题
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]

    # 第一次调用：允许自动选工具
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice='auto',
        max_tokens=800,
    )

    # 取出 assistant message
    msg = response.choices[0].message
    if msg.tool_calls:
        # 若模型要检索：收集每条 tool 的 role=tool 消息
        tool_outputs = []
        for call in msg.tool_calls:
            if call.function.name == 'search_docs':
                # arguments 可能是 JSON 字符串；没有则空 dict
                args = json.loads(call.function.arguments) if hasattr(call.function, 'arguments') else {}
                # 缺省用原 question；k 默认 3
                query = args.get('query', question)
                k = args.get('k', 3)
                # 本地执行检索
                results = search_docs(query, k=k)
                tool_outputs.append({
                    'tool_call_id': call.id,
                    'role': 'tool',
                    'name': 'search_docs',
                    # 多段用空行拼接；没命中则固定英文提示
                    'content': '\n\n'.join(results) if results else 'No relevant sections found.'
                })

        # 第二次调用：messages + 含 tool_calls 的 assistant + tool 结果
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages + [msg] + tool_outputs,
            max_tokens=800,
        )
        # 返回最终结构化回答
        return response.choices[0].message.content

    # 模型没调工具：直接返回 content
    return msg.content


In [ ]:
# ========== 示例提问：演示退款与 Webhook ==========

# 两个英文问题字符串保持原样（影响模型检索词与回答）
print(ask('How do I refund a charge?'))
print(ask('What webhook events exist, and how often are retries?'))


## Short Answer
You refund a charge by making a POST request to the endpoint `/v1/charges/{charge_id}/refunds` with the required amount to refund and optional reason.

## Details
To refund a charge, use the endpoint `POST /v1/charges/{charge_id}/refunds`. You need to specify the amount to refund in cents as a required field. Optionally, you can provide a reason for the refund. The response will include the refund ID and status of the refund.

## Citations
- "Refund Charge (POST /v1/charges/{charge_id}/refunds)"
